# Question-to-Cypher (Q2C) — Execution & Semantic Evaluation

Notebook untuk mengevaluasi akurasi **Question-to-Cypher (Q2C)** pada level hasil eksekusi database.

**Alur Kerja Evaluasi:**
1. Load dataset test case (berisi `TEST_ID`, `QUESTION`, `EXPECTED_CYPHER_QUERY`, `EXPECTED_QUERY_RESULT`).
2. Generate Cypher query untuk setiap `QUESTION` secara langsung menggunakan model LLM dengan prompt dari Google Sheets.
3. Eksekusi Cypher query yang dihasilkan langsung ke Neo4j untuk mendapatkan **Actual Query Result** berupa list of dict (JSON).
4. Bandingkan **Actual Query Result** dengan **Expected Query Result (Ground Truth)** menggunakan normalisasi semantik.
5. Hitung metrik evaluasi:
   - **Syntax/Execution Success Rate**: Persentase query yang berhasil berjalan di Neo4j tanpa error.
   - **Exact Match Rate**: Persentase hasil eksekusi yang sama persis secara konten (tanpa sensitif terhadap urutan kunci/alias).
   - **Result Precision, Recall, dan F1-Score**: Evaluasi tingkat kemiripan data yang ditarik dari graf.
6. Simpan laporan evaluasi ke CSV dan Google Sheets.

In [7]:
# === Setup ===
import sys
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from neo4j import GraphDatabase

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

print(f'Project root: {PROJECT_ROOT}')

Project root: d:\TA\llm-driven-legal-kg-visualization


## Step 0: Configuration

Set `EXPERIMENT_ID` dan sumber test data.

In [8]:
# === Configuration ===
EXPERIMENT_ID = "008"              # Unique ID per eksperimen
PROMPT_ID = "PROMPT_10"             # Prompt version from QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE
SCHEMA_ID = "KG_SCHEMA_3"          # Schema ID from KG_SCHEMA worksheet
DOCUMENT_IDS = ["POJK_11_2022", "UU_11_2008", "UU_19_2016"]

DELAY_BETWEEN_REQUESTS = 1                  # Delay antar request (seconds)

# Sumber test data: 'csv' atau 'gsheets'
TEST_DATA_SOURCE = "csv"                    # 'csv' atau 'gsheets'
CSV_PATH = "HOTS_Q2C_DATATEST_LITE.csv"         # Path file test data Q2C (misal POJK_Q2C_TEST_DATA.csv)
GSHEETS_SHEET_NAME = "HOTS_Q2C_DATATEST_LITE"   # Jika source = gsheets

# Tulis hasil ke Google Sheets?
WRITE_TO_GSHEETS = True
EXPERIMENT_SHEET_NAME = f"EXP_Q2C_{EXPERIMENT_ID}"

print(f'Experiment: {EXPERIMENT_ID}')
print(f'Document IDs: {DOCUMENT_IDS}')
print(f'Test data source: {TEST_DATA_SOURCE}')
print(f'Prompt ID: {PROMPT_ID}')


Experiment: 008
Document IDs: ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016']
Test data source: csv
Prompt ID: PROMPT_10


## Step 1: Load Test Data

In [9]:
# === Load Test Data ===
if TEST_DATA_SOURCE == "gsheets":
    from modules.google_sheets_utils import GoogleUtil
    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
    test_df = gu.load_dataframe_from_sheet(spreadsheet_id, GSHEETS_SHEET_NAME)
else:
    # Look in project root, parent directory, or data directory
    if os.path.exists(CSV_PATH):
        test_df = pd.read_csv(CSV_PATH)
    elif os.path.exists(PROJECT_ROOT / CSV_PATH):
        test_df = pd.read_csv(PROJECT_ROOT / CSV_PATH)
    else:
        test_df = pd.read_csv(PROJECT_ROOT.parent / CSV_PATH)

print(f'Loaded {len(test_df)} test cases')
print(f'Columns: {list(test_df.columns)}')
test_df[['TEST_ID', 'QUESTION', 'CATEGORY', 'EXPECTED_CYPHER_QUERY']].head(5)

Loaded 5 test cases
Columns: ['TEST_ID', 'QUESTION', 'CATEGORY', 'EXPECTED_CYPHER_QUERY', 'EXPECTED_QUERY_RESULT', 'FORMATTED_EXPECTED_QUERY_RESULT']


,TEST_ID,QUESTION,CATEGORY,EXPECTED_CYPHER_QUERY
0,HOTS_001,Bank X mendirikan divisi baru untuk mengelola ...,tata_kelola,MATCH (n:Ayat) WHERE n.source_document_id = 'P...
1,HOTS_002,"Dalam kasus sengketa pinjaman online, nasabah ...",alat_bukti,MATCH (p:Pasal)-[:MEMILIKI_AYAT]->(n:Ayat) WHE...
2,HOTS_003,Sebuah bank umum mengalami kebocoran data nasa...,perlindungan_data,MATCH (n:Ayat) WHERE (n.source_document_id = '...
3,HOTS_004,"Seorang penipu membuat nama domain ""www.klik-b...",manipulasi_data,MATCH (n) WHERE n.source_document_id = 'UU_11_...
4,HOTS_005,Apakah dibenarkan secara hukum apabila sebuah ...,penempatan_data,MATCH (p:Pasal)-[:MEMILIKI_AYAT]->(n:Ayat) WHE...


## Step 1.5: Load Prompt Template from Google Sheets

Mengambil system prompt dan user prompt template yang terdaftar di Google Sheets.

In [10]:
# === Load Prompt from GSheets ===
from modules.prompt_fetcher import fetch_question_to_cypher_prompt
from pipeline.transform.prompt_builder import load_schema_from_gsheets
from modules.google_sheets_utils import GoogleUtil

prompt_data = fetch_question_to_cypher_prompt(PROMPT_ID)

SYSTEM_PROMPT = prompt_data['SYSTEM_PROMPT']
USER_PROMPT_TEMPLATE = prompt_data.get('USER_PROMPT', 'Question: {question}')

# Fetch KG Schema from Google Sheets and format System Prompt
if '{KG_SCHEMA}' in SYSTEM_PROMPT:
    try:
        gu_schema = GoogleUtil(
            private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
            client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
        )
        spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
        kg_schema_str = load_schema_from_gsheets(SCHEMA_ID, gu_schema, spreadsheet_id)
        SYSTEM_PROMPT = SYSTEM_PROMPT.replace('{KG_SCHEMA}', kg_schema_str)
        print("✅ Formatted {KG_SCHEMA} in SYSTEM_PROMPT using Google Sheets (KG_SCHEMA_2)")
    except Exception as e:
        print(f"⚠️ Failed to load KG_SCHEMA from Google Sheets: {e}. Fallback to local config.")
        from pipeline.transform.prompt_builder import load_schema_from_file
        kg_schema_str = load_schema_from_file(PROJECT_ROOT / 'config' / 'kg_schema.json')
        SYSTEM_PROMPT = SYSTEM_PROMPT.replace('{KG_SCHEMA}', kg_schema_str)
        print("✅ Fallback: Formatted {KG_SCHEMA} using config/kg_schema.json")

print(f'System prompt length: {len(SYSTEM_PROMPT)} chars')
print(f'User prompt template length: {len(USER_PROMPT_TEMPLATE)} chars')


2026-06-01 05:17:20,159 - INFO - Fetching prompt 'PROMPT_10' from sheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE'...
2026-06-01 05:17:20,159 - INFO - Retrieving worksheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-06-01 05:17:22,261 - INFO - Successfully loaded 10 rows from worksheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE'.
2026-06-01 05:17:22,264 - INFO - Loaded prompt 'PROMPT_10': ['PROMPT_ID', 'SYSTEM_PROMPT', 'USER_PROMPT', 'NOTES']
2026-06-01 05:17:24,039 - INFO - Loaded schema 'KG_SCHEMA_3' (1358 chars)


✅ Formatted {KG_SCHEMA} in SYSTEM_PROMPT using Google Sheets (KG_SCHEMA_2)
System prompt length: 20142 chars
User prompt template length: 125 chars


## Step 2: Connect to Neo4j

In [11]:
# === Connect to Neo4j ===
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run('RETURN 1 AS test')
    print(f'✅ Connected to Neo4j database: "{NEO4J_DATABASE}"')

✅ Connected to Neo4j database: "experiment-2"


## Step 3: Define Helper Functions

In [12]:
# === Cypher Gen & Execution Helpers ===
import re
import google.generativeai as genai

# Setup Gemini model
genai.configure(api_key=os.getenv('GEMINI_API_KEY', ''))
model = genai.GenerativeModel('gemini-2.5-flash')
print('✅ LLM model ready: gemini-2.5-flash')

def clean_cypher(text: str) -> str:
    """Strip markdown code blocks and extra whitespace from LLM output.
    Matches backend LLMService._clean_cypher() exactly.
    """
    # Remove ```cypher ... ``` or ```sql ... ``` or ``` ... ```
    pattern = r'```(?:cypher|sql|plaintext)?\s*\n?(.*?)```'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Fallback: if starts with ``` but no closing, strip first line
    if text.startswith('```'):
        lines = text.split('\n')
        return '\n'.join(lines[1:]).strip().rstrip('`')
    return text.strip()

def is_valid_cypher(query: str) -> bool:
    """Basic validation: has RETURN, balanced parens/brackets/braces, not empty, and no invalid trailing syntax.
    Matches backend LLMService._is_valid_cypher() exactly.
    """
    if not query or 'RETURN' not in query.upper():
        return False
    if query.count('(') != query.count(')'):
        return False
    if query.count('[') != query.count(']'):
        return False
    if query.count('{') != query.count('}'):
        return False

    # Check for truncated or invalid trailing syntax
    clean_query = query.strip()
    if clean_query.endswith('.') or clean_query.endswith(',') or clean_query.endswith('+') or clean_query.endswith('-'):
        return False

    # Check for trailing logical operators or incomplete clauses
    upper_query = clean_query.upper()
    invalid_trailing_words = [' AND', ' OR', ' WHERE', ' MATCH', ' WITH', ' RETURN']
    for word in invalid_trailing_words:
        if upper_query.endswith(word):
            return False

    return True

def generate_cypher_offline(question: str, doc_ids: list, max_retries: int = 2) -> str:
    """Generate Cypher query from question using direct LLM call.
    Matches backend LLMService.generate_cypher() logic:
    - Uses GSheet prompt template for system prompt
    - Same max_output_tokens (4096) as backend
    - Same retry logic: if first attempt is invalid, retry with simpler prompt
    - Rate-limit handling with exponential backoff
    """
    # Construct doc filter instruction if applicable
    doc_filter_instruction = ''
    if doc_ids:
        ids_str = ', '.join(f"'{d}'" for d in doc_ids)
        doc_filter_instruction = (
            f'\n\nCRITICAL: The user has selected specific document sources. '
            f'You MUST add this filter to your first MATCH clause: '
            f'WHERE <node>.source_document_id IN [{ids_str}]\n'
            f'Apply this to the main entity node (Pasal, Ayat, Bab, or Regulasi) in the query.'
        )

    # Format user prompt from GSheet template
    user_prompt = USER_PROMPT_TEMPLATE
    if '{question}' in user_prompt:
        user_prompt = user_prompt.replace('{question}', question)
    else:
        user_prompt += f'\nQuestion: {question}'

    if '{doc_filter_instruction}' in user_prompt:
        user_prompt = user_prompt.replace('{doc_filter_instruction}', doc_filter_instruction)
    elif doc_filter_instruction:
        user_prompt += doc_filter_instruction

    for attempt in range(max_retries):
        try:
            # Attempt 1: Use full prompt template
            response = model.generate_content(
                [SYSTEM_PROMPT, user_prompt],
                generation_config={'temperature': 0.0, 'max_output_tokens': 4096},
            )
            cypher = clean_cypher(response.text.strip())

            if is_valid_cypher(cypher):
                return cypher

            # Invalid Cypher — retry with simpler prompt (matches backend retry logic)
            print(f'  [attempt {attempt+1}: invalid Cypher, retrying with simpler prompt]')
            retry_prompt = (
                f'Question: {question}\n\n'
                f'Write a SIMPLE Cypher query (1-3 lines) for Neo4j. '
                f'Write MATCH ... RETURN ... LIMIT 100 directly. NO markdown, NO explanation, NO thinking.'
            )
            response = model.generate_content(
                [SYSTEM_PROMPT, retry_prompt],
                generation_config={'temperature': 0.0, 'max_output_tokens': 4096},
            )
            cypher = clean_cypher(response.text.strip())
            if is_valid_cypher(cypher):
                return cypher

        except Exception as e:
            if 'quota' in str(e).lower() or '429' in str(e):
                wait = 5 * (attempt + 1)
                print(f'  [rate limit, waiting {wait}s]', end='')
                time.sleep(wait)
            else:
                raise
    return ''

def run_cypher_query(driver, query: str, database: str) -> list:
    """Execute Cypher query and return results as list of dicts."""
    with driver.session(database=database) as session:
        result = session.run(query)
        return [dict(record) for record in result]


✅ LLM model ready: gemini-2.5-flash


c:\Users\daffarafi\miniconda3\envs\ta-skripsi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\daffarafi\AppData\Local\Temp\ipykernel_26056\2642513030.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Step 4: Define Evaluation Logic (Semantic Matching)

In [13]:
# === Semantic Comparison Logic (Two-Stage: Strict + Flexible) ===
import re

def normalize_value(val) -> str:
    """Normalize value to string for comparison (lowercasing, whitespace stripping)."""
    if val is None:
        return ""
    if isinstance(val, list):
        return "||".join(sorted([normalize_value(item) for item in val]))
    if isinstance(val, dict):
        return "||".join(sorted([f"{k}:{normalize_value(v)}" for k, v in val.items()]))
    if isinstance(val, bool):
        return str(val).lower()
    if isinstance(val, (int, float)):
        if int(val) == val:
            return str(int(val))
        return f"{val:.4f}".rstrip('0').rstrip('.')
    
    s = str(val).strip().lower()
    s = " ".join(s.split())
    s = s.replace('\n', ' ').replace('\r', '').replace('"', '').replace("'", "")
    return s

def record_to_normalized_tuple(record) -> tuple:
    """Convert dict to a sorted tuple of its normalized values, ignoring key names."""
    if not isinstance(record, dict):
        return (normalize_value(record),)
    vals = [normalize_value(v) for v in record.values()]
    return tuple(sorted(vals))

def clean_label(text):
    """Normalize a legal article label for flexible comparison."""
    if not text:
        return ""
    text = str(text).lower().strip()
    text = text.replace('(', '').replace(')', '')
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_legal_labels(record) -> set:
    """Extract legal article/bab labels from a record dict."""
    labels = set()
    if not isinstance(record, dict):
        val = clean_label(record)
        if 'pasal' in val or 'bab' in val:
            labels.add(val)
        return labels
    label_keys = ['pasal_ayat', 'label', 'pasal', 'pasal_larangan', 'pasal_sanksi', 'ayat', 'sumber', 'tujuan', 'bab']
    for k in label_keys:
        if k in record and record[k]:
            val = clean_label(record[k])
            if 'pasal' in val or 'bab' in val or len(val) > 3:
                labels.add(val)
    if not labels:
        for v in record.values():
            val = clean_label(v)
            if 'pasal' in val or 'bab' in val:
                labels.add(val)
    return labels

def evaluate_execution_results(actual_list: list, expected_val) -> dict:
    """Two-stage evaluation: strict tuple match first, then flexible legal-label match."""
    # Parse expected values
    expected_list = []
    if expected_val:
        if isinstance(expected_val, str):
            try:
                expected_list = json.loads(expected_val)
            except Exception:
                expected_list = [{"expected": expected_val}]
        elif isinstance(expected_val, list):
            expected_list = expected_val
        else:
            expected_list = [expected_val]

    if not expected_list and not actual_list:
        return {"passed": True, "precision": 1.0, "recall": 1.0, "f1": 1.0, "notes": "Both empty"}
    if not expected_list:
        return {"passed": False, "precision": 0.0, "recall": 1.0, "f1": 0.0, "notes": "Expected empty, got results"}
    if not actual_list:
        return {"passed": False, "precision": 1.0, "recall": 0.0, "f1": 0.0, "notes": "Expected results, got empty"}

    # --- STAGE 1: Strict Normalized Tuple Comparison ---
    actual_tuples = set(record_to_normalized_tuple(r) for r in actual_list)
    expected_tuples = set(record_to_normalized_tuple(r) for r in expected_list)
    intersection_strict = actual_tuples.intersection(expected_tuples)
    p_s = len(intersection_strict) / len(actual_tuples) if actual_tuples else 0.0
    r_s = len(intersection_strict) / len(expected_tuples) if expected_tuples else 0.0
    f1_s = 2 * p_s * r_s / (p_s + r_s) if (p_s + r_s) > 0 else 0.0

    if f1_s == 1.0:
        return {"passed": True, "precision": 1.0, "recall": 1.0, "f1": 1.0, "notes": "Exact match (Strict)"}

    # --- STAGE 2: Flexible Legal Label Comparison ---
    expected_labels = set()
    for r in expected_list:
        expected_labels.update(extract_legal_labels(r))
    actual_labels = set()
    for r in actual_list:
        actual_labels.update(extract_legal_labels(r))
    expected_labels = {l for l in expected_labels if len(l) > 3}
    actual_labels = {l for l in actual_labels if len(l) > 3}

    if not expected_labels:
        return {
            "passed": f1_s >= 0.9,
            "precision": round(p_s, 4), "recall": round(r_s, 4), "f1": round(f1_s, 4),
            "notes": f"No labels, strict fallback. {len(intersection_strict)}/{len(expected_tuples)} shared"
        }

    shared = expected_labels.intersection(actual_labels)
    p_f = len(shared) / len(actual_labels) if actual_labels else 0.0
    r_f = len(shared) / len(expected_labels) if expected_labels else 0.0
    f1_f = 2 * p_f * r_f / (p_f + r_f) if (p_f + r_f) > 0 else 0.0
    passed = expected_labels.issubset(actual_labels)

    return {
        "passed": passed,
        "precision": round(max(p_s, p_f), 4),
        "recall": round(max(r_s, r_f), 4),
        "f1": round(max(f1_s, f1_f), 4),
        "notes": f"Flexible Stage. Expected: {expected_labels}, Shared: {shared}, Recall: {r_f:.2%}"
    }


## Step 5: Run Evaluation

In [14]:
# === Run Evaluation Loop ===
results = []
total_cases = len(test_df)

print(f"Starting Q2C Execution Evaluation on {total_cases} test cases...\n")

for i, row in test_df.iterrows():
    test_id = row['TEST_ID']
    question = row['QUESTION']
    category = row['CATEGORY']
    expected_result_raw = row.get('EXPECTED_QUERY_RESULT') or row.get('FORMATTED_EXPECTED_QUERY_RESULT') or ""
    
    print(f"[{i+1}/{total_cases}] Running {test_id} ({category})...")
    
    # Step 1: Generate Cypher query directly using Google Sheets prompt template
    try:
        generated_cypher = generate_cypher_offline(question, DOCUMENT_IDS)
        if not generated_cypher:
            raise ValueError("Empty or invalid Cypher generated")
    except Exception as e:
        generated_cypher = ""
        print(f"  ❌ Generation Error: {e}")
        print("  ❌ FAIL: Cypher generation failed")
        results.append({
            "TEST_ID": test_id,
            "QUESTION": question,
            "CATEGORY": category,
            "EXPECTED_CYPHER": row.get("EXPECTED_CYPHER_QUERY", ""),
            "GENERATED_CYPHER": "",
            "STATUS": "GEN_ERROR",
            "EXEC_SUCCESS": False,
            "PRECISION": 0.0,
            "RECALL": 0.0,
            "F1": 0.0,
            "ERROR": str(e),
            "ACTUAL_RESULT": "[]",
            "FORMATTED_ACTUAL_QUERY_RESULT": "[]"
        })
        continue
        
    print(f"  Generated Cypher: {generated_cypher}")
    
    # Step 2: Execute query in Neo4j
    exec_success = False
    actual_results = []
    error_msg = ""
    
    try:
        actual_results = run_cypher_query(driver, generated_cypher, NEO4J_DATABASE)
        exec_success = True
        print(f"  Execution: Success ({len(actual_results)} rows)")
    except Exception as e:
        error_msg = str(e)
        print(f"  ❌ Execution Error: {error_msg[:100]}...")
        
    # Step 3: Evaluate results
    if exec_success:
        eval_metrics = evaluate_execution_results(actual_results, expected_result_raw)
        status = "PASS" if eval_metrics["passed"] else "MISMATCH"
        precision = eval_metrics["precision"]
        recall = eval_metrics["recall"]
        f1 = eval_metrics["f1"]
        notes = eval_metrics["notes"]
        print(f"  Evaluation: {status} (F1={f1:.2%})")
    else:
        status = "EXEC_ERROR"
        precision = 0.0
        recall = 0.0
        f1 = 0.0
        notes = f"Execution error: {error_msg}"
        
    results.append({
        "TEST_ID": test_id,
        "QUESTION": question,
        "CATEGORY": category,
        "EXPECTED_CYPHER": row.get("EXPECTED_CYPHER_QUERY", ""),
        "GENERATED_CYPHER": generated_cypher,
        "STATUS": status,
        "EXEC_SUCCESS": exec_success,
        "PRECISION": precision,
        "RECALL": recall,
        "F1": f1,
        "ERROR": error_msg,
        "ACTUAL_RESULT": json.dumps(actual_results, ensure_ascii=False),
        "FORMATTED_ACTUAL_QUERY_RESULT": json.dumps(actual_results, indent=2, ensure_ascii=False),
        "NOTES": notes
    })
    
    # Delay to avoid overloading
    time.sleep(DELAY_BETWEEN_REQUESTS)

results_df = pd.DataFrame(results)
print("\n=== Evaluation Completed! ===")

Starting Q2C Execution Evaluation on 5 test cases...

[1/5] Running HOTS_001 (tata_kelola)...
  Generated Cypher: MATCH (b:Bab)-[:MEMUAT*1..2]->(p:Pasal) WHERE b.source_document_id = 'POJK_11_2022' AND toLower(b.label) CONTAINS 'tata kelola ti bank' OPTIONAL MATCH (p)-[:MEMILIKI_AYAT]->(a:Ayat) RETURN b.label AS bab, p.label AS pasal, p.content AS isi_pasal, a.label AS ayat, a.content AS isi_ayat, b.source_document_id AS regulasi ORDER BY p.label, a.label LIMIT 150
  Execution: Success (21 rows)
  Evaluation: PASS (F1=7.14%)
[2/5] Running HOTS_002 (alat_bukti)...
  Generated Cypher: MATCH (n)
WHERE n.source_document_id IN ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016']
  AND (n:Pasal OR n:Ayat)
  AND (toLower(n.content) CONTAINS 'informasi elektronik' OR toLower(n.content) CONTAINS 'dokumen elektronik')
  AND (toLower(n.content) CONTAINS 'bukti' OR toLower(n.content) CONTAINS 'alat bukti' OR toLower(n.content) CONTAINS 'sah')
RETURN labels(n) AS tipe, n.label AS label, n.content AS isi, n

## Step 6: Summary & Breakdown

In [15]:
# === Overall Summary ===
total = len(results_df)
gen_success = len(results_df[results_df['STATUS'] != 'GEN_ERROR'])
exec_success = len(results_df[results_df['EXEC_SUCCESS'] == True])
exact_match = len(results_df[results_df['STATUS'] == 'PASS'])
avg_f1 = results_df['F1'].mean()

print(f'╔══════════════════════════════════════════════════════════════╗')
print(f'║  Q2C Execution Evaluation Report                             ║')
print(f'╠══════════════════════════════════════════════════════════════╣')
print(f'║  Experiment: {EXPERIMENT_ID:<47s} ║')
print(f'║  Database:   {NEO4J_DATABASE:<47s} ║')
print(f'╠══════════════════════════════════════════════════════════════╣')
print(f'║  Total Cases:       {total:>3d}                                      ║')
print(f'║  Generation Success: {gen_success:>3d}  ({gen_success/total:.1%})                             ║')
print(f'║  Execution Success:  {exec_success:>3d}  ({exec_success/total:.1%})                             ║')
print(f'║  Exact Match (Pass): {exact_match:>3d}  ({exact_match/total:.1%})                             ║')
print(f'║  Average F1-Score:   {avg_f1:.2%}                                  ║')
print(f'╚══════════════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════════════╗
║  Q2C Execution Evaluation Report                             ║
╠══════════════════════════════════════════════════════════════╣
║  Experiment: 008                                             ║
║  Database:   experiment-2                                    ║
╠══════════════════════════════════════════════════════════════╣
║  Total Cases:         5                                      ║
║  Generation Success:   5  (100.0%)                             ║
║  Execution Success:    5  (100.0%)                             ║
║  Exact Match (Pass):   1  (20.0%)                             ║
║  Average F1-Score:   11.26%                                  ║
╚══════════════════════════════════════════════════════════════╝


In [16]:
# === Per-Category Breakdown ===
print(f'\n{"CATEGORY":<20s} {"TOTAL":>5s} {"GEN_OK":>7s} {"EXEC_OK":>8s} {"PASS":>5s} {"AVG_F1":>8s}')
print('─' * 60)

for cat in results_df['CATEGORY'].unique():
    cat_df = results_df[results_df['CATEGORY'] == cat]
    cat_total = len(cat_df)
    cat_gen = len(cat_df[cat_df['STATUS'] != 'GEN_ERROR'])
    cat_exec = len(cat_df[cat_df['EXEC_SUCCESS'] == True])
    cat_pass = len(cat_df[cat_df['STATUS'] == 'PASS'])
    cat_f1 = cat_df['F1'].mean()
    print(f'{cat:<20s} {cat_total:>5d} {cat_gen:>7d} {cat_exec:>8d} {cat_pass:>5d} {cat_f1:>8.2%}')

print('─' * 60)
print(f'{"TOTAL":<20s} {total:>5d} {gen_success:>7d} {exec_success:>8d} {exact_match:>5d} {avg_f1:>8.2%}')


CATEGORY             TOTAL  GEN_OK  EXEC_OK  PASS   AVG_F1
────────────────────────────────────────────────────────────
tata_kelola              1       1        1     1    7.14%
alat_bukti               1       1        1     0   40.00%
perlindungan_data        1       1        1     0    3.92%
manipulasi_data          1       1        1     0    5.26%
penempatan_data          1       1        1     0    0.00%
────────────────────────────────────────────────────────────
TOTAL                    5       5        5     1   11.26%


## Step 7: Display Failed / Mismatch Cases

In [17]:
# === Failed / Mismatch Cases Detail ===
mismatch_df = results_df[results_df['STATUS'].isin(['MISMATCH', 'EXEC_ERROR', 'GEN_ERROR'])]

if len(mismatch_df) == 0:
    print("🎉 Congratulations! All queries passed with 100% exact match.")
else:
    print(f"❌ Found {len(mismatch_df)} non-passing cases:\n")
    for _, row in mismatch_df.head(15).iterrows():
        print(f"Test ID: {row['TEST_ID']} ({row['CATEGORY']})")
        print(f"Question: {row['QUESTION']}")
        print(f"Generated Cypher: {row['GENERATED_CYPHER']}")
        print(f"Status: {row['STATUS']}, F1: {row['F1']:.2%}")
        if row['ERROR']:
            print(f"Error: {row['ERROR']}")
        else:
            print(f"Notes: {row['NOTES']}")
        print("-" * 50)

❌ Found 4 non-passing cases:

Test ID: HOTS_002 (alat_bukti)
Question: Dalam kasus sengketa pinjaman online, nasabah mengajukan tangkapan layar (screenshot) percakapan WhatsApp sebagai bukti pelunasan utang. Pihak kreditur menyangkal bukti tersebut karena bukan dokumen cetak resmi. Bagaimana kekuatan hukum tangkapan layar tersebut sebagai alat bukti di persidangan?
Generated Cypher: MATCH (n)
WHERE n.source_document_id IN ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016']
  AND (n:Pasal OR n:Ayat)
  AND (toLower(n.content) CONTAINS 'informasi elektronik' OR toLower(n.content) CONTAINS 'dokumen elektronik')
  AND (toLower(n.content) CONTAINS 'bukti' OR toLower(n.content) CONTAINS 'alat bukti' OR toLower(n.content) CONTAINS 'sah')
RETURN labels(n) AS tipe, n.label AS label, n.content AS isi, n.source_document_id AS regulasi
LIMIT 100
Status: MISMATCH, F1: 40.00%
Notes: Flexible Stage. Expected: {'pasal 5 ayat 3', 'pasal 5 ayat 4', 'pasal 5 ayat 1', 'pasal 5 ayat 2'}, Shared: {'pasal 5 ayat 3',

## Step 8: Save Laporan Evaluasi

In [18]:
# === Save Results ===
output_dir = 'data/evaluation'
os.makedirs(output_dir, exist_ok=True)

output_csv = f'{output_dir}/q2c_exec_eval_{EXPERIMENT_ID}.csv'
results_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f'✅ Results saved to: {output_csv}')

if WRITE_TO_GSHEETS:
    try:
        from modules.google_sheets_utils import GoogleUtil, GoogleSheetsWriter
        gu = GoogleUtil(
            private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
            client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
        )
        spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
        writer = GoogleSheetsWriter(
            google_util=gu,
            sheet_id=spreadsheet_id,
            worksheet_name=EXPERIMENT_SHEET_NAME,
            batch_size=5,
        )
        writer.write_dataframe(results_df)
        print(f'✅ Results uploaded to Google Sheets: {EXPERIMENT_SHEET_NAME}')
    except Exception as e:
        print(f"⚠️ Failed to upload to Google Sheets: {e}")

✅ Results saved to: data/evaluation/q2c_exec_eval_008.csv


  0%|          | 0/1 [00:00<?, ?it/s]

2026-06-01 05:19:15,654 - ERROR - Error writing row 1: EXP_Q2C_008
2026-06-01 05:19:16,905 - ERROR - Error writing row 2: EXP_Q2C_008
2026-06-01 05:19:18,141 - ERROR - Error writing row 3: EXP_Q2C_008
2026-06-01 05:19:19,428 - ERROR - Error writing row 4: EXP_Q2C_008
2026-06-01 05:19:20,569 - ERROR - Error writing row 5: EXP_Q2C_008
100%|██████████| 1/1 [00:06<00:00,  6.23s/it]

✅ Results uploaded to Google Sheets: EXP_Q2C_008


In [19]:
# === Cleanup ===
driver.close()
print("✅ Neo4j connection driver closed.")

✅ Neo4j connection driver closed.
